This code will convert python program into different languages like C#, Java, JS, Go and Rust.
User can select what language he wants to convert this code from the dropdown menu.

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from system_info import retrieve_system_info

openai = OpenAI()

openai_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = 'gemini-2.5-flash'
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(api_key=openai_api_key, base_url=gemini_url)

In [ ]:
def system_prompt_for(language):
    f"""
Your task is to convert Python code into {language} code.
Respond only with one {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output and response should only contain one solution.
"""

def user_prompt_for(python, language):
    return f"""
Port this Python code to {language} with the implementation that produces identical output in the least time.
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [ ]:

LANGUAGES = ["C#", "C++", "Java", "JavaScript", "Go", "Rust"]

COMPILER_LINKS = {
    "C#": "https://dotnetfiddle.net/",
    "C++": "https://www.onlinegdb.com/online_c++_compiler",
    "Java": "https://www.jdoodle.com/online-java-compiler/",
    "JavaScript": "https://jsfiddle.net/",
    "Go": "https://play.golang.org/",
    "Rust": "https://play.rust-lang.org/"
}

In [ ]:
def messages_for(python, language):
    return [
        {"role": "system", "content": system_prompt_for(language)},
        {"role": "user", "content": user_prompt_for(python, language)}
    ]

def port(python, language):
    response = openai.chat.completions.create(model=MODEL, messages=messages_for(python, language))
    reply = response.choices[0].message.content
    return reply, f'<a href="{COMPILER_LINKS[language]}" target="_blank">Run {language} code online</a>'

In [ ]:
pi = """
import time

def fibonacci(n):
    a, b = 0, 1
    series = []
    for _ in range(n):
        series.append(a)
        a, b = b, a + b
    return series

result = fibonacci(10)

print(f"Result: {result}")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
from styles import CSS
import gradio as gr

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to other coding languages") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=pi,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"Generated code",
                value="",
                language="cpp",
                lines=26
            )


    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])

        language_dropdown = gr.Dropdown(
            choices=LANGUAGES,
            value="C#",
            label="Select Target Language"
        )

        convert = gr.Button(f"Port Now", elem_classes=["convert-btn"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            compiler_link = gr.HTML(label="Online Compiler")

    convert.click(fn=port, inputs=[python, language_dropdown], outputs=[cpp, compiler_link])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])

ui.launch(inbrowser=True)